# 00 — Colab Setup

**Run this notebook ONCE before any training notebook.**

This notebook handles:
1. GPU verification
2. Google Drive mount
3. Repository clone
4. Dependencies installation
5. Data symlinks
6. Complete environment verification

**Author:** Chadha Jeddi — NILM Benchmarking Project  
**Target:** Google Colab T4/A100 GPU

## ⚠️ Before Running

Make sure you have:
1. **GPU enabled**: Runtime → Change runtime type → T4 GPU → Save
2. **Data on Google Drive** at:
```
My Drive/nilm_project/data/raw/UKDALE/ukdale.h5       (3.3 GB)
My Drive/nilm_project/data/raw/REDD/redd.h5           (401 MB)
My Drive/nilm_project/data/raw/AMPds2/AMPds2/         (folder)
My Drive/nilm_project/data/raw/REFIT/CLEAN_House*.csv (20 files)
```

## Cell 1 — GPU Verification

In [ ]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU:             {gpu}")
    print(f"GPU memory:      {mem:.1f} GB")
    print("\n✅ GPU ready for training")
else:
    print("\n❌ No GPU detected!")
    print("Go to: Runtime → Change runtime type → GPU (T4) → Save")
    raise RuntimeError("GPU required. Please enable GPU and restart.")

## Cell 2 — Mount Google Drive

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Verify drive is mounted
drive_root = '/content/drive/MyDrive'
assert os.path.exists(drive_root), "Drive mount failed!"
print(f"✅ Google Drive mounted at {drive_root}")

# Set base path for all NILM data on Drive
DRIVE_NILM = f"{drive_root}/nilm_project"
os.makedirs(DRIVE_NILM, exist_ok=True)
print(f"✅ NILM project folder: {DRIVE_NILM}")

## Cell 3 — Verify Dataset Files on Drive

In [ ]:
import os

DRIVE_NILM = '/content/drive/MyDrive/nilm_project'

datasets = {
    'UK-DALE': f"{DRIVE_NILM}/data/raw/UKDALE/ukdale.h5",
    'REDD':    f"{DRIVE_NILM}/data/raw/REDD/redd.h5",
    'AMPds2':  f"{DRIVE_NILM}/data/raw/AMPds2/AMPds2/Electricity/Sub_meter_data/Electricity_WHE.csv",
    'REFIT':   f"{DRIVE_NILM}/data/raw/REFIT/CLEAN_House1.csv",
}

all_ok = True
print("Dataset verification:")
for name, path in datasets.items():
    exists = os.path.exists(path)
    if exists:
        size = os.path.getsize(path) / 1e6
        print(f"  ✅ {name:<10} {size:.0f} MB  {path}")
    else:
        print(f"  ❌ {name:<10} NOT FOUND: {path}")
        all_ok = False

if all_ok:
    print("\n✅ All datasets found on Google Drive")
else:
    print("\n⚠️ Some datasets missing. Upload them to Drive before training.")
    print("Training will work for available datasets only.")

## Cell 4 — Clone GitHub Repository

In [ ]:
import os
import subprocess

REPO_DIR = '/content/nilm-benchmarking'
REPO_URL = 'https://github.com/chadhajeddi-ux/nilm-benchmarking'

# Remove old clone if exists
if os.path.exists(REPO_DIR):
    print("Removing old clone...")
    os.system(f'rm -rf {REPO_DIR}')

# Clone repository
print(f"Cloning {REPO_URL}...")
result = os.system(f'git clone {REPO_URL} {REPO_DIR}')

if result == 0:
    print(f"\n✅ Repository cloned to {REPO_DIR}")
    os.chdir(REPO_DIR)
    print(f"Working directory: {os.getcwd()}")
    # Show repo contents
    print("\nRepository structure:")
    os.system('ls -la')
else:
    # Try with token if repo is private
    print("\n❌ Clone failed. If repo is private, set your token:")
    print("GITHUB_TOKEN = 'ghp_your_token_here'")
    print("Then run: git clone https://{GITHUB_TOKEN}@github.com/chadhajeddi-ux/nilm-benchmarking")

## Cell 4b — If Repository is Private (run only if Cell 4 failed)

In [ ]:
# Only run this cell if the repo is private and Cell 4 failed
# Replace with your actual GitHub token

GITHUB_TOKEN = 'ghp_your_token_here'  # ← paste your token here
REPO_DIR = '/content/nilm-benchmarking'

if os.path.exists(REPO_DIR):
    os.system(f'rm -rf {REPO_DIR}')

os.system(
    f'git clone https://{GITHUB_TOKEN}@github.com/chadhajeddi-ux/nilm-benchmarking {REPO_DIR}'
)
os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

## Cell 5 — Install Dependencies

In [ ]:
import subprocess
import sys

print("Installing dependencies from requirements.txt...")
print("(torch is pre-installed on Colab, skipping)\n")

# Install everything except torch (already on Colab)
packages = [
    'PyWavelets>=1.4.0',
    'pyarrow>=12.0.0',
    'h5py>=3.8.0',
    'tqdm>=4.65.0',
    'einops>=0.6.0',
    'omegaconf>=2.3.0',
    'torchinfo>=1.8.0',
    'openpyxl>=3.1.0',
    'seaborn>=0.12.0',
]

for pkg in packages:
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', pkg],
        capture_output=True, text=True
    )
    status = '✅' if result.returncode == 0 else '❌'
    print(f"  {status} {pkg}")

print("\n✅ All packages installed")

## Cell 6 — Setup Python Path

In [ ]:
import sys
import os

REPO_DIR = '/content/nilm-benchmarking'

# Add all necessary paths
paths = [
    f'{REPO_DIR}/src',
    f'{REPO_DIR}/models',
    f'{REPO_DIR}/models/baselines',
    f'{REPO_DIR}/models/proposed',
]

for path in paths:
    if path not in sys.path:
        sys.path.insert(0, path)
    print(f"  ✅ Added: {path}")

os.chdir(REPO_DIR)
print(f"\n✅ Python path configured")
print(f"Working directory: {os.getcwd()}")

## Cell 7 — Create Directory Structure

In [ ]:
import os

REPO_DIR = '/content/nilm-benchmarking'
DRIVE_NILM = '/content/drive/MyDrive/nilm_project'

# Create directories in repo
local_dirs = [
    'data/raw/UKDALE',
    'data/raw/REDD',
    'data/raw/AMPds2',
    'data/raw/REFIT',
    'data/processed',
]

for d in local_dirs:
    os.makedirs(f'{REPO_DIR}/{d}', exist_ok=True)

# Create persistent folders on Drive
drive_dirs = [
    'experiments/checkpoints',
    'experiments/results',
    'data/processed',
]

for d in drive_dirs:
    os.makedirs(f'{DRIVE_NILM}/{d}', exist_ok=True)

print("✅ Directory structure created")

# Create symlinks: repo → Drive (so results persist after session)
symlinks = {
    # Local path in repo → Drive path
    f'{REPO_DIR}/data/raw/UKDALE/ukdale.h5':       f'{DRIVE_NILM}/data/raw/UKDALE/ukdale.h5',
    f'{REPO_DIR}/data/raw/REDD/redd.h5':           f'{DRIVE_NILM}/data/raw/REDD/redd.h5',
    f'{REPO_DIR}/data/raw/AMPds2/AMPds2':          f'{DRIVE_NILM}/data/raw/AMPds2/AMPds2',
}

# REFIT CSVs — symlink the entire folder
refit_src = f'{DRIVE_NILM}/data/raw/REFIT'
refit_dst = f'{REPO_DIR}/data/raw/REFIT'
if os.path.exists(refit_src) and not os.path.islink(refit_dst):
    if os.path.exists(refit_dst):
        import shutil
        shutil.rmtree(refit_dst)
    os.symlink(refit_src, refit_dst)
    print(f"  ✅ Linked REFIT folder")

print("\nCreating symlinks for data files:")
for local, drive_path in symlinks.items():
    if os.path.exists(drive_path):
        if not os.path.exists(local) and not os.path.islink(local):
            os.symlink(drive_path, local)
            print(f"  ✅ Linked: {os.path.basename(local)}")
        else:
            print(f"  ✅ Already exists: {os.path.basename(local)}")
    else:
        print(f"  ⚠️ Not on Drive: {os.path.basename(local)}")

# Symlink experiments folder to Drive (results persist!)
for folder in ['checkpoints', 'results']:
    src = f'{DRIVE_NILM}/experiments/{folder}'
    dst = f'{REPO_DIR}/experiments/{folder}'
    if os.path.exists(dst) and not os.path.islink(dst):
        import shutil
        shutil.rmtree(dst)
    if not os.path.islink(dst):
        os.symlink(src, dst)
        print(f"  ✅ Linked experiments/{folder} → Drive (persistent!)")

# Symlink processed data cache to Drive
proc_src = f'{DRIVE_NILM}/data/processed'
proc_dst = f'{REPO_DIR}/data/processed'
if os.path.exists(proc_dst) and not os.path.islink(proc_dst):
    import shutil
    shutil.rmtree(proc_dst)
if not os.path.islink(proc_dst):
    os.symlink(proc_src, proc_dst)
    print(f"  ✅ Linked data/processed → Drive (caches parquet files!)")

print("\n✅ All symlinks created")

## Cell 8 — Verify All Imports

In [ ]:
print("Verifying all imports...\n")

# Core modules
try:
    from config import (
        WINDOW_SIZE, INPUT_CHANNELS, N_APPLIANCES,
        APPLIANCE_NAMES, APPLIANCES, BATCH_SIZE,
        RESULTS_DIR, CHECKPOINTS_DIR, DATASETS
    )
    print(f"✅ config.py")
    print(f"   INPUT_CHANNELS={INPUT_CHANNELS} | WINDOW={WINDOW_SIZE} | N_APP={N_APPLIANCES}")
    print(f"   Datasets: {list(DATASETS.keys())}")
except Exception as e:
    print(f"❌ config.py: {e}")

try:
    from dwt import dwt_transform
    import numpy as np
    test = dwt_transform(np.random.randn(480).astype(np.float32))
    assert test.shape == (4, 480)
    print(f"✅ dwt.py — output shape {test.shape}")
except Exception as e:
    print(f"❌ dwt.py: {e}")

try:
    from preprocessing import (
        load_ukdale_house, load_redd_house,
        load_ampds_house, load_refit_house, preprocess_house
    )
    print(f"✅ preprocessing.py — all 4 dataset loaders")
except Exception as e:
    print(f"❌ preprocessing.py: {e}")

try:
    from dataset import NILMDataset, NormStats, split_train_val, build_dataloaders
    print(f"✅ dataset.py")
except Exception as e:
    print(f"❌ dataset.py: {e}")

try:
    from metrics import MetricsTracker, multi_task_loss, focal_loss
    print(f"✅ metrics.py")
except Exception as e:
    print(f"❌ metrics.py: {e}")

try:
    from train import get_model_registry, train_model, load_data
    registry = get_model_registry()
    print(f"✅ train.py — {len(registry)} models: {list(registry.keys())}")
except Exception as e:
    print(f"❌ train.py: {e}")

try:
    from evaluate import evaluate, load_test_data
    print(f"✅ evaluate.py")
except Exception as e:
    print(f"❌ evaluate.py: {e}")

## Cell 9 — Verify All Models

In [ ]:
import torch
from train import get_model_registry
from config import INPUT_CHANNELS, WINDOW_SIZE, N_APPLIANCES

registry = get_model_registry()
x_test = torch.randn(4, INPUT_CHANNELS, WINDOW_SIZE).cuda()

print(f"Testing all {len(registry)} models on GPU...")
print(f"Input shape: {tuple(x_test.shape)}\n")
print(f"{'Model':<16} {'Params':>10}  {'INT8 MB':>8}  {'Output':>12}  Status")
print("-" * 60)

all_pass = True
for name, model_class in registry.items():
    try:
        model = model_class().cuda()
        model.eval()
        with torch.no_grad():
            p, s, g = model(x_test)
        n = sum(par.numel() for par in model.parameters())
        ok = p.shape == (4, N_APPLIANCES)
        status = '✅' if ok else '❌'
        print(f"{name:<16} {n:>10,}  {n/1e6:>8.3f}  {str(tuple(p.shape)):>12}  {status}")
        if not ok:
            all_pass = False
        del model
        torch.cuda.empty_cache()
    except Exception as e:
        print(f"{name:<16} ERROR: {e}")
        all_pass = False

print()
if all_pass:
    print("✅ All models verified on GPU — ready for training!")
else:
    print("❌ Some models failed — check errors above")

## Cell 10 — Save Setup Config (for other notebooks)

In [ ]:
import json
import os
import torch

REPO_DIR = '/content/nilm-benchmarking'
DRIVE_NILM = '/content/drive/MyDrive/nilm_project'

# Save setup configuration for other notebooks to load
setup_config = {
    "repo_dir": REPO_DIR,
    "drive_nilm": DRIVE_NILM,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
    "python_paths": [
        f'{REPO_DIR}/src',
        f'{REPO_DIR}/models',
        f'{REPO_DIR}/models/baselines',
        f'{REPO_DIR}/models/proposed',
    ],
    "data_paths": {
        "ukdale": f"{REPO_DIR}/data/raw/UKDALE/ukdale.h5",
        "redd":   f"{REPO_DIR}/data/raw/REDD/redd.h5",
        "ampds2": f"{REPO_DIR}/data/raw/AMPds2",
        "refit":  f"{REPO_DIR}/data/raw/REFIT",
    },
    "experiments": {
        "checkpoints": f"{REPO_DIR}/experiments/checkpoints",
        "results":     f"{REPO_DIR}/experiments/results",
    }
}

# Save to Drive (persists between sessions)
config_path = f"{DRIVE_NILM}/colab_setup.json"
with open(config_path, 'w') as f:
    json.dump(setup_config, f, indent=2)

print(f"✅ Setup config saved to Drive: {config_path}")
print()
print("Setup complete! You can now run any training notebook.")
print()
print("Notebook order:")
print("  01_train_cnn.ipynb")
print("  02_train_gru.ipynb")
print("  03_train_bigru.ipynb")
print("  04_train_lstm.ipynb")
print("  05_train_bilstm.ipynb")
print("  06_train_cnn_lstm.ipynb")
print("  07_train_nilmformer.ipynb")
print("  08_train_biwave.ipynb  ← most important")
print("  09_comparison.ipynb")
print("  10_deployment.ipynb")

## ✅ Setup Complete

Your environment is ready. All subsequent training notebooks use a single helper function to initialize:

```python
# Copy this at the top of every training notebook
import json, sys, os
cfg = json.load(open('/content/drive/MyDrive/nilm_project/colab_setup.json'))
for p in cfg['python_paths']:
    sys.path.insert(0, p)
os.chdir(cfg['repo_dir'])
```

**Checkpoints and results are saved to Google Drive automatically** — they persist even after the Colab session ends.